# Лабораторная работа № 2. Проектирование, обучение и проверка нейросетевой модели

**90 минут.** Продолжение первой работы: формулы прямого прохода и backprop уже знакомы,
поэтому цикл обучения здесь предоставлен. Основная задача — спроектировать эксперимент,
выбрать модель и обосновать вывод о ее качестве.

После занятия вы сможете: задать размеры сети; подготовить X и D; разделить роли L/T/P;
выбрать ширину скрытого слоя; распознать насыщение и другие причины проблем;
организовать перекрестную проверку без утечки; объяснить смысл свертки и полносвязного слоя.

**Среда:** Python 3, Jupyter, NumPy, Matplotlib, уже установленные до занятия.
Интернет и скачивание данных не требуются. Данные синтетические и встроены.

**Порядок работы:** выполняйте ячейки сверху вниз, заполняйте `TODO` и поля «Ответ».
Ячейки с `NotImplementedError` остановятся до заполнения. Не переписывайте готовую инфраструктуру.
Отдельный длинный отчет не нужен: сдайте этот notebook с кодом, выводом и краткими объяснениями.

**Важное соглашение.** В лекции T используют для выбора архитектуры: в современной
терминологии оно выполняет роль validation. P — отложенный final test. P открываем
только после фиксации всех настроек. Это согласование терминов, а не изменение лекционной схемы.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 23
np.set_printoptions(precision=4, suppress=True)
print('Среда готова:', np.__version__)


## A. Постановка задачи и архитектура

Моделируем синтетическую зависимость: два измеренных входа и одно вещественное значение.
В генераторе выход зависит от первого входа; второй независим и добавлен как пример
потенциально лишнего признака. В реальных данных такая информация заранее может быть неизвестна.

**1.** Задайте X и D словами. Почему Nx=2 и Ny=1? Чем проектирование отличается от обучения?
Перечислите этапы разработки нейросетевой модели.

**2.** Для сети Nx–h–Ny посчитайте число связей без смещений и общее число параметров:
$$N_{weights}=N_xh+hN_y,\qquad N_{params}=N_xh+h+hN_y+N_y.$$

**3.** Рассчитайте лекционные ориентиры при Nx=2, Ny=1, Q=32:
$$\frac{N_yQ}{1+\log_2 Q}\leq N_w\leq N_y\left(\frac{Q}{N_x}+1\right)(N_x+N_y+1)+N_y.$$


In [ ]:
def parameter_counts(nx, h, ny):
    # TODO: число связей без смещений и число всех параметров
    raise NotImplementedError('Допишите две формулы')

for h in [1, 4, 16, 64]:
    weights, total = parameter_counts(2, h, 1)
    print(f'2–{h}–1: связей={weights}, параметров со смещениями={total}')
assert parameter_counts(2, 4, 1) == (12, 17)

# Лекционные формулы; код готов, требуется интерпретация.
nx, ny, q = 2, 1, 32
lower = ny*q/(1+np.log2(q))
upper = ny*(q/nx+1)*(nx+ny+1)+ny
print('Лекционный интервал Nw:', lower, upper)
print('Ориентир h без учета смещений:', lower/(nx+ny), upper/(nx+ny))


X=…; D=…; Nx=…; Ny=… . Проектирование / обучение: … .
От чего зависит подходящая ширина скрытого слоя и активация? … .
Этапы создания модели: … . Лекционные ориентиры Nw и h: … .


## B. Обучающие данные и предобработка

Создайте непересекающиеся L/T/P размером 32/48/80. Случайное разделение подходит
для нашего независимого синтетического набора; для временных или групповых данных нужна другая схема.
Разбиение фиксируем до обучения и сравнения моделей.


In [ ]:
# Основные данные. Шум измерения цели — часть постановки, а не ошибка маркировки.
rng = np.random.default_rng(SEED)
n = 160
latent = rng.uniform(-1, 1, (n, 2))
X_raw = np.column_stack((2000+1000*latent[:, 0], 300+80*latent[:, 1]))
d_raw = (100+30*(np.sin(np.pi*latent[:, 0])+0.15*latent[:, 0])
         + rng.normal(0, 3, n)).reshape(-1, 1)
order = rng.permutation(n)

# TODO: разделите order на первые 32, следующие 48 и оставшиеся 80 индексов.
L, T, P = None, None, None


В отдельном маленьком примере выявите одинаковые X с разными d и кандидата в выбросы.
Это пример аудита, а не испорченный основной датасет. Не удаляйте записи автоматически:
уточните источник, возможный недостающий признак и смысл редкого наблюдения.

In [ ]:
assert (len(L), len(T), len(P)) == (32, 48, 80)
assert len(np.unique(np.concatenate([L, T, P]))) == n
assert not set(L) & set(T) and not set(L) & set(P) and not set(T) & set(P)
plt.scatter(X_raw[L, 0], d_raw[L, 0], label='L')
plt.xlabel('Первый вход, исходные единицы'); plt.ylabel('Цель, исходные единицы')
plt.title('Обучающие примеры'); plt.grid(); plt.legend(); plt.show()

# Отдельный пример контроля качества, не используемый для обучения основной сети.
X_audit = np.array([[2,4], [2,4], [3,1], [4,2], [5,3]], dtype=float)
d_audit = np.array([4., 5., 6., 8., 80.])
def conflict_groups(X, d):
    groups = {}
    for i, x in enumerate(X): groups.setdefault(tuple(x), []).append(i)
    # TODO: оставить группы одинаковых X с более чем одним уникальным d.
    raise NotImplementedError('Выявите конфликтные группы')

conflicts = conflict_groups(X_audit, d_audit)
assert conflicts == [[0, 1]]
print('Конфликтные записи:', conflicts)
# Для этой учебной схемы известен допустимый диапазон цели [0,12].
suspects = np.flatnonzero((d_audit < 0) | (d_audit > 12))
print('Кандидаты на проверку источника:', suspects)


Допишите преобразование в [A,B] и обратное преобразование:
$$\tilde x=\frac{x-x_{min}}{x_{max}-x_{min}}(B-A)+A,$$
$$x=\frac{\tilde x-A}{B-A}(x_{max}-x_{min})+x_{min}.$$
Минимумы и максимумы признаков и цели оцениваются **только по L**. На T/P применяются
те же параметры. Новое значение может выйти за [A,B]; не пересчитывайте границы по T/P.

На выходе используем функцию Ле-Кана с диапазоном (−1.7159,1.7159),
поэтому цели из L отображаем в [−1,1] с запасом до насыщения.

In [ ]:
def scale_to(A, lo, hi, left=-1., right=1.):
    span = np.asarray(hi)-np.asarray(lo)
    safe_span = np.where(span == 0, 1., span)
    # TODO: линейное отображение; постоянные столбцы отобразятся в left.
    raise NotImplementedError('Допишите нормализацию')

def inverse_scale(Z, lo, hi, left=-1., right=1.):
    # TODO: обратное преобразование
    raise NotImplementedError('Допишите денормализацию')


In [ ]:
x_lo, x_hi = X_raw[L].min(axis=0), X_raw[L].max(axis=0)
d_lo, d_hi = d_raw[L].min(axis=0), d_raw[L].max(axis=0)
# P пока не преобразуем и не оцениваем.
X_L = scale_to(X_raw[L], x_lo, x_hi); X_T = scale_to(X_raw[T], x_lo, x_hi)
D_L = scale_to(d_raw[L], d_lo, d_hi); D_T = scale_to(d_raw[T], d_lo, d_hi)
assert np.allclose(X_L.min(0), -1) and np.allclose(X_L.max(0), 1)
assert np.allclose(inverse_scale(D_L, d_lo, d_hi), d_raw[L])
assert np.allclose(scale_to(np.array([5.]), 5., 5.), [-1.])
assert np.allclose(inverse_scale(np.array([0.]), 5., 5.), [5.])
print('Нормализация и обратное преобразование проверены')
print('Эмпирический ориентир лекции Q=7Nx+15:', 7*X_L.shape[1]+15)


- L используется для …; T — для …; P — для … .
- Почему конфликты не исчезают от увеличения числа нейронов? Для целей 4 и 5
  при одинаковом X какое общее предсказание минимизирует MSE и чему равна эта MSE?
- Как дополнительный значимый признак может разрешить конфликт? Когда он не поможет?
- Что проверять перед удалением выброса? Почему падение обучающей ошибки недостаточно?
- Почему параметры масштабирования нельзя оценивать по всему датасету?
- Назовите источники пар Xq–Dq: наблюдения, архивы, эксперименты, моделирование и др.
  Какая особенность есть у синтетических данных?
- Почему Q=7Nx+15 не гарантирует достаточно данных? Как формируются категориальные X и несколько выходов D?

**Ответ:** …


## C. Размер сети, обучение и обобщение

Используем функцию Ле-Кана $\varphi(s)=1.7159\tanh(2s/3)$:
$$H=\varphi(XW_1+b_1),\qquad Y=\varphi(HW_2+b_2).$$
Цель обучения — уменьшить $\frac1{2N}\sum_n(Y_n-D_n)^2$ на L.
В таблицах сообщаем **MSE без множителя 1/2**; в конце — RMSE в исходных единицах.
Все сравнения внутри этапа используют одинаковую нормировку, разбиение, число шагов и seed.

Допишите активацию и ее производную. Цикл backprop и моментум уже предоставлены.


In [ ]:
LECUN_A, LECUN_B = 1.7159, 2/3
def activation(s):
    # TODO: функция Ле-Кана
    raise NotImplementedError('Допишите activation')

def activation_derivative(s):
    # TODO: производная по s
    raise NotImplementedError('Допишите производную')


Запустите h=1,4,16,64; допишите вычисление ошибок и выбор h по T.
Для каждой сети сохраняется лучшая по T контрольная эпоха. P не используется.

In [ ]:
assert np.allclose(activation(np.array([-1.,0.,1.])), [-1.,0.,1.], atol=1e-4)
eps = 1e-6; z_check = np.array([-1.5, 0., 1.5])
assert np.allclose(activation_derivative(z_check),
                   (activation(z_check+eps)-activation(z_check-eps))/(2*eps))

def mse(y, d): return float(np.mean((np.asarray(y)-np.asarray(d))**2))
def predict(X, params):
    W1,b1,W2,b2 = params
    return activation(activation(X@W1+b1)@W2+b2)

def augment_inputs(X, sigma, rng):
    # Готово пока без добавления шума. В блоке E функция будет переопределена.
    return np.asarray(X).copy()

def fit_mlp(X, D, h=4, eta=.03, steps=1800, seed=SEED, validation=None, noise=0.):
    rng = np.random.default_rng(seed)
    W1=rng.normal(0, 1/np.sqrt(X.shape[1]), (X.shape[1],h)); b1=np.zeros(h)
    W2=rng.normal(0, 1/np.sqrt(h), (h,D.shape[1])); b2=np.zeros(D.shape[1])
    params=[W1,b1,W2,b2]
    velocity=[np.zeros_like(p) for p in params]
    best=None; best_loss=np.inf; history=[]
    for epoch in range(steps):
        Xi=augment_inputs(X, noise, rng) if noise else X
        s1=Xi@W1+b1; H=activation(s1); s2=H@W2+b2; Y=activation(s2)
        delta2=(Y-D)*activation_derivative(s2)/len(X)
        delta1=(delta2@W2.T)*activation_derivative(s1)
        grads=[Xi.T@delta1, delta1.sum(0), H.T@delta2, delta2.sum(0)]
        for p,v,g in zip(params,velocity,grads):
            v[:]=.9*v+g
            p-=eta*v
        if epoch%20==0 or epoch==steps-1:
            loss_l=mse(predict(X,params),D)
            loss_t=np.nan if validation is None else mse(predict(validation[0],params),validation[1])
            history.append((epoch+1,loss_l,loss_t))
            if validation is not None and loss_t<best_loss:
                best_loss=loss_t; best=[p.copy() for p in params]
    if validation is None:best=[p.copy() for p in params]
    if best is None:raise ValueError('Проверьте конечность ошибки и параметры обучения')
    return best, np.array(history)


In [ ]:
architectures = [1, 4, 16, 64]
models = {}; histories = {}
for h in architectures:
    models[h], histories[h] = fit_mlp(X_L,D_L,h=h,validation=(X_T,D_T))

def model_errors(params):
    # TODO: ошибки MSE на L и T; P здесь запрещено использовать.
    raise NotImplementedError('Вычислите две ошибки')

errors = np.array([model_errors(models[h]) for h in architectures])
# TODO: выберите h с минимальной ошибкой T (второй столбец errors).
best_h = None


Сравните ошибки L/T и сложность; проверьте влияние перестановки входных признаков.
Кривые ошибок не обязаны быть монотонными или иметь учебную U-форму на любом запуске.

In [ ]:
assert best_h == architectures[int(np.argmin(errors[:,1]))]
print('h / параметры / MSE_L / MSE_T')
for h,(e_l,e_t) in zip(architectures,errors):
    print(h,parameter_counts(2,h,1)[1],round(e_l,5),round(e_t,5))
print('Выбрана ширина:',best_h)
fig,axes=plt.subplots(1,2,figsize=(11,3))
axes[0].plot(architectures,errors[:,0],'o-',label='L')
axes[0].plot(architectures,errors[:,1],'o-',label='T')
axes[0].set_xscale('log',base=2);axes[0].set_xticks(architectures,labels=architectures)
axes[0].set_xlabel('Скрытых нейронов h');axes[0].set_ylabel('MSE');axes[0].legend();axes[0].grid()
for h in architectures:axes[1].plot(histories[h][:,0],histories[h][:,2],label=f'h={h}')
axes[1].set_xlabel('Эпоха');axes[1].set_ylabel('MSE на T');axes[1].legend();axes[1].grid()
plt.tight_layout();plt.show()

# Увеличение ошибки после перестановки — чувствительность модели, не доказательство причинности.
base_t = mse(predict(X_T,models[best_h]),D_T)
for j in range(X_T.shape[1]):
    altered=X_T.copy()
    altered[:,j]=np.random.default_rng(900+j).permutation(altered[:,j])
    print(f'Признак {j}: прирост MSE_T={mse(predict(altered,models[best_h]),D_T)-base_t:.4f}')


- Чем отличаются ошибка обучения и ошибка обобщения? Почему h выбираем по T, а не по L?
- Какие h дают признаки недообучения или переобучения в этом опыте? На что, кроме h, влияет результат?
- Опишите примерные лекционные кривые L/T; совпал ли с ними ваш график?
- Чем проверка сети отличается от оптимизации ее архитектуры и обучения весов?
- Как выявлять значимые признаки: веса, возмущение входа, исключение и повторное обучение?
  Почему малый вес сам по себе недостаточен? Что может произойти при исключении нужного входа?
- В13: аналогия «идеальное запоминание учебных ответов против переноса знания» объясняет
  переобучение. Можно ли из нее вывести утверждение об успехе людей с разными школьными оценками?

**Ответ:** …


## D. Методы обучения и причины неудач

**1.** Сравните функцию Ле-Кана и «логарифмическую» функцию лекции:
$$\psi(s)=\ln(s+\sqrt{s^2+1})=\operatorname{arsinh}(s),\qquad
\psi'(s)=\frac1{\sqrt{1+s^2}}.$$
Ее удобно и устойчиво вычислять через `np.arcsinh`. У нее нет ограниченного диапазона,
но производная тоже уменьшается при больших |s|. Она не гарантирует устранение всех проблем обучения.

**2.** Посмотрите, что происходит с производными скрытых нейронов при исходных
ненормализованных входах. Сопоставьте кривые обучения при разных η и начальных seed.

**3.** Объясните шесть причин, по которым сеть может не обучаться удовлетворительно: ...

**Сравнение методов.** Градиентный шаг использует величину градиента; RPROP — его знак
и адаптивный шаг по каждому весу; квазиньютоновские методы приближенно учитывают кривизну.
Обычный backprop вычисляет градиент, а выбранный оптимизатор использует его.


In [ ]:
z=np.linspace(-8,8,401)
fig,axes=plt.subplots(1,2,figsize=(10,3))
axes[0].plot(z,activation(z),label='Ле-Кан');axes[0].plot(z,np.arcsinh(z),label='arsinh')
axes[0].set_xlabel('s');axes[0].set_ylabel('y');axes[0].legend();axes[0].grid()
axes[1].plot(z,activation_derivative(z),label='Ле-Кан: производная')
axes[1].plot(z,1/np.sqrt(1+z*z),label='arsinh: производная')
axes[1].set_xlabel('s');axes[1].set_ylabel('dy/ds');axes[1].legend();axes[1].grid()
plt.tight_layout();plt.show()

W_probe=np.random.default_rng(SEED).normal(0,1/np.sqrt(2),(2,4))
for name,X in [('Исходные входы',X_raw[L]),('Нормализованные',X_L)]:
    derivative=activation_derivative(X@W_probe)
    print(name,': средняя производная=',derivative.mean(),'; доля < 1e-6=',np.mean(derivative<1e-6))

fig,ax=plt.subplots(figsize=(6,3))
for eta in [.005,.03,.15]:
    _,trace=fit_mlp(X_L,D_L,h=best_h,eta=eta,steps=600,validation=(X_T,D_T))
    ax.plot(trace[:,0],trace[:,1],label=f'η={eta}')
ax.set_xlabel('Эпоха');ax.set_ylabel('MSE_L');ax.grid();ax.legend();plt.show()

restart_models={};restart_scores={}
for seed in [23,24,25]:
    restart_models[seed],_=fit_mlp(X_L,D_L,h=best_h,seed=seed,validation=(X_T,D_T))
    restart_scores[seed]=mse(predict(X_T,restart_models[seed]),D_T)
print('Разные инициализации, MSE_T:',restart_scores)
# Выбираем по T; обучающая ошибка или P не используются для выбора seed.
best_seed=min(restart_scores,key=restart_scores.get)
print('Выбранный seed:',best_seed)


- Какая активация насыщается? Почему маленькая производная замедляет обучение?
- Можно ли по плохому результату доказать, что найден именно локальный минимум?
- Что использует RPROP, а что учитывает квазиньютоновский метод?
- Чем отличаются повторный старт и изменение архитектуры?
- Перечислите шесть причин и по одному действию для их проверки.
- Почему перемешивание примеров важно для SGD, но не меняет полный пакетный градиент?


## E. Шум, перекрестная проверка и подтверждение

Допишите добавление независимого нормального шума к **входам L**.
σ здесь означает стандартное отклонение, а дисперсия равна σ².
Сравните σ=0,0.03,0.20 при одинаковых прочих настройках. Не изменяйте T/P и не обещайте,
что любой шум улучшит качество. Допустимая величина зависит от задачи и масштаба.


In [ ]:
def augment_inputs(X, sigma, rng):
    # TODO: X + независимый шум N(0, sigma²) той же формы
    raise NotImplementedError('Допишите добавление шума')

assert np.array_equal(augment_inputs(X_L,0,np.random.default_rng(1)),X_L)
noise_models={};noise_scores={}
for sigma in [0.,.03,.20]:
    noise_models[sigma],_=fit_mlp(X_L,D_L,h=best_h,seed=best_seed,noise=sigma,validation=(X_T,D_T))
    noise_scores[sigma]=mse(predict(X_T,noise_models[sigma]),D_T)
print('Шум / MSE_T:',noise_scores)
best_sigma=min(noise_scores,key=noise_scores.get)
print('Выбранный σ:',best_sigma)


Допишите разбиение для 4-fold проверки на пуле L∪T.
Это компактный учебный вариант, а не буквально десять повторов из лекции.
Для CV заранее фиксируем h=4 и 800 эпох; в каждом фолде заново обучаем масштабирование
и сеть только по обучающей части. Валидационный фолд не выбирает лучшую эпоху.
CV здесь демонстрирует отдельный фиксированный протокол, а не оценивает без смещения
весь уже проведенный поиск архитектуры/seed. Для оценки выбранной модели сохраняем P.

In [ ]:
def make_folds(size,k=4,seed=501):
    shuffled=np.random.default_rng(seed).permutation(size)
    # TODO: непересекающиеся части, вместе покрывающие shuffled
    raise NotImplementedError('Допишите разбиение на фолды')

pool=np.concatenate([L,T])
folds=make_folds(len(pool))
assert len(folds)==4 and len(np.unique(np.concatenate(folds)))==len(pool)
assert sum(len(f) for f in folds)==len(pool)
cv_scores=[]
for fold_id,valid_local in enumerate(folds):
    train_local=np.setdiff1d(np.arange(len(pool)),valid_local)
    train_ids,valid_ids=pool[train_local],pool[valid_local]
    lo,hi=X_raw[train_ids].min(0),X_raw[train_ids].max(0)
    dlo,dhi=d_raw[train_ids].min(0),d_raw[train_ids].max(0)
    xtr=scale_to(X_raw[train_ids],lo,hi);xva=scale_to(X_raw[valid_ids],lo,hi)
    dtr=scale_to(d_raw[train_ids],dlo,dhi)
    params,_=fit_mlp(xtr,dtr,h=4,steps=800,seed=SEED,validation=None)
    pred_raw=inverse_scale(predict(xva,params),dlo,dhi)
    cv_scores.append(np.sqrt(mse(pred_raw,d_raw[valid_ids])))
print('CV: RMSE по фолдам в исходных единицах:',np.array(cv_scores))
print('Средняя RMSE:',np.mean(cv_scores),'максимальная:',np.max(cv_scores))


Зафиксируйте h, seed, σ и сохраненные веса, затем один раз оцените P.
После просмотра P настройки не меняйте; в конце notebook повторного выбора по P нет.

In [ ]:
# Все решения приняты до обращения к P.
selected_params=[p.copy() for p in noise_models[best_sigma]]
frozen_settings={'h':best_h,'seed':best_seed,'sigma':best_sigma}
print('Зафиксированные настройки:',frozen_settings)
X_P=scale_to(X_raw[P],x_lo,x_hi)
Y_P_raw=inverse_scale(predict(X_P,selected_params),d_lo,d_hi)
Y_T_raw=inverse_scale(predict(X_T,selected_params),d_lo,d_hi)
rmse_t=np.sqrt(mse(Y_T_raw,d_raw[T]));rmse_p=np.sqrt(mse(Y_P_raw,d_raw[P]))
print('RMSE_T:',rmse_t,'; RMSE_P:',rmse_p,'(исходные единицы цели)')


- Как изменился результат при добавлении шума? Что ограничивает этот вывод?
- Почему каждый фолд требует нового масштабирования и обучения?
- Чем 4-fold проверка отличается от одного разделения и от описанных в лекции повторов?
- Почему P дает более независимую оценку после выбора по T? Каковы оставшиеся ограничения?
- Можно ли обойтись без P и какой протокол нужен для независимой оценки после подбора?

Итоговые h/seed/σ и $RMSE_P$: … .


## F. Декомпозиция, классификация и свертка

Для Nx=2, h=4, Ny=3 сравните общую сеть и три отдельных сети с одним выходом.
Почему раздельные модели иногда удобны, а иногда теряют полезное совместное представление?
Улучшение не гарантировано и должно проверяться на T.


In [ ]:
shared=parameter_counts(2,4,3)[1]
separate=3*parameter_counts(2,4,1)[1]
print('Общая сеть: параметров=',shared,'; три отдельных=',separate)
assert (shared,separate)==(27,51)
for score in [.9,.4,1.2,-.1]:
    if not 0<=score<=1:print(score,'— вне диапазона логистического выхода')
    else:print(score,'— класс 1' if score>.5 else '— класс 0')


Для сигмоидного выхода классификатора интерпретируйте 0.9 и 0.4 при пороге >0.5.

**Ответ:** ...

Допишите локальную фильтрацию изображения ядром 3×3. В нейросетевой практике
операцию без переворота ядра часто называют сверткой; математически это взаимная корреляция.
Одинаковый фильтр применяется во всех позициях. Сравните valid и заполнение нулями:
пространственный размер уменьшается не всегда.

Затем полносвязный слой получает развернутую карту и дает два числа-оценки классов.
Это иллюстрация прямого прохода с заданными весами, **не обученный классификатор изображений**.
Более глубокие слои могут строить более сложные признаки, но это нельзя проверить одним фильтром.

In [ ]:
def filter_valid(image,kernel):
    kh,kw=kernel.shape
    out=np.empty((image.shape[0]-kh+1,image.shape[1]-kw+1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            patch=image[i:i+kh,j:j+kw]
            # TODO: скалярное произведение окна и ядра без переворота
            raise NotImplementedError('Вычислите значение фильтра')
    return out

image=np.array([[0,0,1,1,1], [0,0,1,1,1], [0,0,1,1,1], [0,0,1,1,1], [0,0,1,1,1]],dtype=float)
kernel=np.array([[-1,0,1],[-1,0,1],[-1,0,1]],dtype=float)
feature=filter_valid(image,kernel)
feature_same=filter_valid(np.pad(image,1),kernel)
assert feature.shape==(3,3) and feature_same.shape==image.shape
assert np.allclose(feature,[[3,3,0],[3,3,0],[3,3,0]])
fig,axes=plt.subplots(1,3,figsize=(9,3))
for ax,matrix,title in zip(axes,[image,feature,feature_same],['Вход 5×5','Фильтр valid: 3×3','С заполнением: 5×5']):
    ax.imshow(matrix,cmap='coolwarm');ax.set_title(title);ax.set_xticks([]);ax.set_yticks([])
plt.tight_layout();plt.show()

flat=feature.ravel()
# Заданные демонстрационные веса головы классификации.
W_dense=np.column_stack((np.ones(flat.size)/flat.size,-np.ones(flat.size)/flat.size))
scores=1/(1+np.exp(-(flat@W_dense)))
print('Полносвязный слой: 9 признаков → 2 оценки:',scores)
print('Выбор одного класса по argmax:',int(np.argmax(scores)))

Общая сеть / отдельные сети: параметры … / …; преимущества и ограничения: … .
Чем выход классификатора отличается от гарантированной вероятности правильного ответа? … .
Что выделил фильтр? Уменьшает ли свертка размер всегда? … .
Что делает полносвязный слой с извлеченными признаками? … .


## G. Итоговый контроль

1. Для сети 3–5–2 найдите число параметров со смещениями. Чем проектирование отличается от обучения?
2. Объясните роли L/T/P и почему нельзя выбирать h по P. Где оценивают параметры масштабирования?
3. Назовите два признака переобучения; почему конфликтные записи и насыщение не лечатся одним увеличением h?
4. Что нужно заново обучать в каждом фолде? Где допустимо добавлять тренировочный шум?
5. Объясните роль фильтра и полносвязной головы. Почему значение выхода 0.9 не гарантирует 90% правильных ответов?

**Ответы 1–5:** …

**Итог 2–3 предложения:** выбранная конфигурация и $RMSE_P$; что проверили;
какие выводы ограничены размером данных и правилами генерации? …

После оценки P не меняйте настройки. Если хотите продолжить исследования,
нужен новый независимый финальный набор или отдельный корректный протокол.


## Дополнительно

- Провести повторные разбиения или вложенную CV для всего поиска h/seed/σ.
- Исключить второй вход, заново обучить модель и проверить ошибку T.
- Заменить пакетное обучение на SGD и сравнить порядок предъявления примеров.
- Реализовать RPROP и сравнить с предоставленным моментумом при одинаковом протоколе.
- Проверить численно градиенты всех весов маленькой сети.
- Исследовать другой закон генерации и новые независимые данные.
